# Lesson 10B: Topic Modeling Practical — Gensim, scikit-learn, and pyLDAvis

## Introduction

Lesson 10A derived collapsed Gibbs sampling from scratch on a small three-topic corpus — enough to prove the algorithm correct, not what you'd reach for on a real document collection. This lesson moves to the production tools: **Gensim**'s `LdaModel` (the standard library for topic modeling in Python), **scikit-learn**'s `LatentDirichletAllocation` for a second implementation, and **pyLDAvis** for the interactive visualization that makes topic models actually usable by a human reader.

The other question 10A left open: **how many topics, $K$?** Unlike K-Means or GMM, there's no elbow-on-inertia here — the standard practical answer is **topic coherence**, a metric that scores how semantically related a topic's top words are to each other, computed without any ground truth.

In this lesson:
1. Build a larger, richer synthetic corpus (5 topics instead of 3)
2. Fit Gensim's `LdaModel` and inspect recovered topics
3. Fit scikit-learn's `LatentDirichletAllocation` for comparison
4. Choose $K$ using topic coherence (`c_v`) across a range of candidate values
5. Visualize the fitted model interactively with `pyLDAvis`
6. Interpret topic coherence and its limits

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [A Richer Synthetic Corpus](#dataset)
4. [Gensim LdaModel](#gensim-lda)
5. [scikit-learn LatentDirichletAllocation](#sklearn-lda)
6. [Choosing K with Topic Coherence](#coherence)
7. [Interactive Visualization with pyLDAvis](#pyldavis)
8. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis
import pyLDAvis.gensim_models

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="dataset"></a>
## A Richer Synthetic Corpus

Five topics this time, each with its own characteristic vocabulary, generated the same way as 10A (draw $\theta_d \sim \text{Dirichlet}(\alpha)$ per document, then sample each word's topic and identity) — larger and more varied than 10A's three-topic corpus, closer to what a real topic-discovery task looks like.

In [ ]:
rng = np.random.RandomState(42)

topic_vocab = {
    'sports': ['ball', 'game', 'team', 'score', 'player', 'coach', 'match', 'win', 'league', 'season'],
    'cooking': ['recipe', 'oven', 'bake', 'flavor', 'ingredient', 'chef', 'dish', 'spice', 'kitchen', 'meal'],
    'finance': ['stock', 'market', 'invest', 'profit', 'bank', 'trade', 'shares', 'fund', 'economy', 'growth'],
    'technology': ['software', 'algorithm', 'computer', 'data', 'network', 'device', 'code', 'server', 'app', 'cloud'],
    'travel': ['flight', 'hotel', 'passport', 'destination', 'luggage', 'tourist', 'itinerary', 'airport', 'visa', 'journey'],
}
true_topic_names = list(topic_vocab.keys())
K_true = len(true_topic_names)
vocab = sorted({w for words in topic_vocab.values() for w in words})
V = len(vocab)
word_to_idx = {w: i for i, w in enumerate(vocab)}

alpha_true, beta_true = 0.3, 0.1
phi_true = np.zeros((K_true, V))
for k, topic in enumerate(true_topic_names):
    for w in topic_vocab[topic]:
        phi_true[k, word_to_idx[w]] = 1.0
    phi_true[k] += beta_true
    phi_true[k] /= phi_true[k].sum()

n_docs, doc_length = 500, 50
docs_tokens = []
for d in range(n_docs):
    theta_d = rng.dirichlet(np.full(K_true, alpha_true))
    words = []
    for _ in range(doc_length):
        z = rng.choice(K_true, p=theta_d)
        w = rng.choice(V, p=phi_true[z])
        words.append(vocab[w])
    docs_tokens.append(words)

print(f"Corpus: {n_docs} documents, vocabulary size {V}, {K_true} true generating topics")
print(f"Example document tokens: {docs_tokens[0][:12]}...")

<a name="gensim-lda"></a>
## Gensim LdaModel

Gensim's workflow: build a `Dictionary` (vocabulary + id mapping), convert each document to a bag-of-words representation (`doc2bow`), then fit `LdaModel`. This is the standard production entry point for topic modeling in Python — used far more often in practice than hand-rolled Gibbs sampling.

In [ ]:
dictionary = corpora.Dictionary(docs_tokens)
bow_corpus = [dictionary.doc2bow(doc) for doc in docs_tokens]

lda_gensim = LdaModel(corpus=bow_corpus, id2word=dictionary, num_topics=K_true, random_state=42, passes=10, alpha='auto')

print("Gensim LdaModel — top words per topic:")
for topic_id, topic_terms in lda_gensim.print_topics(num_words=8):
    print(f"  Topic {topic_id}: {topic_terms}")

<a name="sklearn-lda"></a>
## scikit-learn LatentDirichletAllocation

Same corpus, scikit-learn's API instead: `CountVectorizer` builds the document-term matrix, `LatentDirichletAllocation` fits the model. A second independent implementation, useful when a project is already built around the scikit-learn ecosystem rather than Gensim's.

In [ ]:
documents_as_strings = [' '.join(doc) for doc in docs_tokens]
vectorizer = CountVectorizer(vocabulary=vocab)
dtm = vectorizer.fit_transform(documents_as_strings)

lda_sklearn = LatentDirichletAllocation(n_components=K_true, random_state=42, max_iter=20, learning_method='batch')
lda_sklearn.fit(dtm)
phi_sklearn = lda_sklearn.components_ / lda_sklearn.components_.sum(axis=1, keepdims=True)

print("scikit-learn LatentDirichletAllocation — top words per topic:")
for k in range(K_true):
    top_idx = np.argsort(-phi_sklearn[k])[:8]
    print(f"  Topic {k}: {[vocab[i] for i in top_idx]}")

<a name="coherence"></a>
## Choosing K with Topic Coherence

`c_v` coherence scores a topic by how often its top words co-occur across sliding windows of the corpus, combined with a normalized pointwise mutual information similarity — in short, **higher coherence means the topic's top words genuinely tend to appear together**, which is what "a topic makes sense" should mean. We fit models across a range of $K$ and inspect how coherence responds — without ever looking at our known ground truth of 5 topics.

In [ ]:
candidate_k = list(range(2, 9))
coherence_scores = []

for k in candidate_k:
    model = LdaModel(corpus=bow_corpus, id2word=dictionary, num_topics=k, random_state=42, passes=5, alpha='auto')
    cm = CoherenceModel(model=model, texts=docs_tokens, dictionary=dictionary, coherence='c_v', processes=1)
    coherence_scores.append(cm.get_coherence())
    print(f"K={k}: coherence={coherence_scores[-1]:.4f}")

best_k = candidate_k[int(np.argmax(coherence_scores))]
print(f"\nArgmax K by raw coherence: {best_k} (true generating K was {K_true})")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(candidate_k, coherence_scores, marker='o', linewidth=2)
ax.axvline(K_true, color='green', linestyle='--', label=f'True K={K_true}')
ax.axvline(best_k, color='red', linestyle=':', label=f'Argmax K={best_k}')
ax.set_xlabel('Number of topics (K)')
ax.set_ylabel('c_v coherence')
ax.set_title('Topic Coherence vs Number of Topics', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### A Caveat: Coherence Doesn't Always Peak at the True K

The argmax above lands on a K well past the true generating K=5 — and this isn't a bug in the
metric or the run. `c_v` coherence rewards topics whose top words co-occur tightly, and
**splitting a topic into narrower sub-topics can only ever tighten that co-occurrence further**
— so coherence often keeps climbing (or at least doesn't reliably fall) as K grows, well past
the point where the extra topics are semantically meaningful. Blindly maximizing coherence is
the topic-modeling equivalent of always picking the largest K in K-Means because inertia keeps
dropping.

**Practical response**: look for a **plateau or a clear local peak** rather than the global
argmax, read the actual top words at each candidate K before committing, and treat coherence as
one input alongside domain judgment about how many distinct themes the corpus plausibly
contains — not a fully automated answer.

<a name="pyldavis"></a>
## Interactive Visualization with pyLDAvis

`pyLDAvis` projects topics onto two dimensions by inter-topic distance (bubble position and size = topic prevalence) and shows each topic's most relevant terms on hover — the standard way to explore a fitted topic model interactively rather than reading `print_topics` output.

Note: pyLDAvis's default topic-distance projection (classical MDS) can produce complex-valued intermediate results with recent NumPy/SciPy versions that break JSON serialization — we use `mds='mmds'` (metric MDS via scikit-learn) instead, which is numerically stable and visually equivalent for this purpose.

In [ ]:
pyLDAvis.enable_notebook()

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=FutureWarning)  # sklearn MDS `init` default-change notice, not an error
    lda_vis = pyLDAvis.gensim_models.prepare(lda_gensim, bow_corpus, dictionary, mds='mmds')

lda_vis

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Gensim and scikit-learn both provide production-ready LDA** — `Dictionary` + `doc2bow` + `LdaModel` for Gensim, `CountVectorizer` + `LatentDirichletAllocation` for scikit-learn. Neither requires hand-rolling Gibbs sampling or variational inference.
2. **Topic coherence (`c_v`) gives a label-free way to score candidate $K$ values** — how often each topic's top words co-occur across the corpus — but the raw argmax over $K$ is not a reliable answer on its own, as this lesson's own run demonstrated.
3. **Coherence tends to keep climbing as $K$ grows** — narrower topics co-occur more tightly by construction, so the global argmax over-splits; look for a plateau or local peak and sanity-check the actual top words instead of trusting the maximum blindly.
4. **pyLDAvis turns a fitted model into something a non-technical stakeholder can explore** — topic size, inter-topic distance, and relevant terms, all interactively.
5. **Library defaults can break on unrelated version changes** (pyLDAvis's classical-MDS path here) — always have a fallback (`mds='mmds'`) rather than assuming the documented default always works.

### Practical Guidance

- Start model selection with a coherence sweep across a reasonable range of $K$, not a single guess.
- Read the top 8-10 words per topic manually even after checking coherence — a metric is a filter, not a replacement for judgment.
- Use pyLDAvis early in exploration, not just at the end — it surfaces topic overlap and imbalance that a `print_topics` list hides.

### Next Steps

This closes Lesson 10. In Lesson 11, we turn to self-organizing maps: competitive learning for topology-preserving 2D projections.

You now have the complete topic-modeling toolkit: the generative model and inference derivation from 10A, and the production workflow — Gensim, scikit-learn, coherence-based model selection, and interactive visualization — from this lesson 🎯